# 06 - WC2026 Results Presentation

Ce notebook presente les resultats de simulation (qualif, phases finales, vainqueur) avec des visualisations claires.

In [47]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "vscode"

DATA_PATH = Path("../data/processed/wc2026_simulation.csv")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df.head()

,team,qual_prob,r16_prob,qf_prob,sf_prob,final_prob,win_prob
0,Argentina,0.954,0.769,0.547,0.340,0.222,0.147
1,France,0.956,0.734,0.497,0.330,0.197,0.120
2,Belgium,0.975,0.760,0.522,0.334,0.194,0.114
3,Brazil,0.952,0.728,0.496,0.320,0.181,0.111
4,Spain,0.897,0.634,0.439,0.255,0.157,0.083


## Resume rapide

In [48]:
summary = df[["qual_prob", "r16_prob", "qf_prob", "sf_prob", "final_prob", "win_prob"]].describe()
summary

,qual_prob,r16_prob,qf_prob,sf_prob,final_prob,win_prob
count,48.000000,48.000000,48.000000,48.000000,48.000000,48.000000
mean,0.666667,0.333333,0.166667,0.083333,0.041667,0.020833
std,0.184199,0.205222,0.154411,0.101010,0.060868,0.037143
min,0.353000,0.089000,0.014000,0.001000,0.000000,0.000000
25%,0.513000,0.163000,0.055000,0.016000,0.006000,0.001000
50%,0.668500,0.293500,0.106000,0.037000,0.012000,0.003000
75%,0.810500,0.482250,0.237500,0.093250,0.041250,0.016750
max,0.975000,0.769000,0.547000,0.340000,0.222000,0.147000


In [49]:
import numpy as np

# On charge les données
df = pd.read_csv(DATA_PATH)

# Normalisation rapide pour la visualisation (on ramène le max à 100%)
cols_to_norm = ["qual_prob", "r16_prob", "qf_prob", "sf_prob", "final_prob", "win_prob"]
for col in cols_to_norm:
    if df[col].max() > 1:
        df[col] = df[col] / df[col].max()

# On trie par probabilité de victoire
df_top = df.sort_values("win_prob", ascending=False).head(15)

In [50]:
fig_win = px.bar(
    df_top.head(10),
    x="win_prob",
    y="team",
    orientation='h',
    title="<b>Top 10 - Probabilité de Victoire Finale (WC 2026)</b>",
    labels={"win_prob": "Probabilité de titre", "team": "Équipe"},
    color="win_prob",
    color_continuous_scale="Viridis",
    text_auto='.1%'
)
fig_win.update_layout(yaxis={'categoryorder':'total ascending'}, showlegend=False)
fig_win.show()

In [51]:
# Préparation des données pour la heatmap
heatmap_data = df_top.set_index("team")[cols_to_norm]
heatmap_data.columns = ["Poules", "8èmes", "Quarts", "Demis", "Finale", "Vainqueur"]

fig_heat = px.imshow(
    heatmap_data,
    labels=dict(x="Stade de la compétition", y="Équipe", color="Probabilité"),
    x=heatmap_data.columns,
    y=heatmap_data.index,
    color_continuous_scale="RdYlGn",
    title="<b>Chances de survie par étape</b>",
    aspect="auto"
)
fig_heat.update_xaxes(side="top")
fig_heat.show()

In [52]:
fig_scatter = px.scatter(
    df_top,
    x="qual_prob",
    y="win_prob",
    text="team",
    size="win_prob",
    color="win_prob",
    title="<b>Stabilité en poules vs Capacité à conclure</b>",
    labels={"qual_prob": "Chances de sortir des poules", "win_prob": "Chances de titre"},
    hover_name="team"
)
fig_scatter.update_traces(textposition='top center')
fig_scatter.show()

In [53]:
fig_map = px.choropleth(
    df,
    locations="team",
    locationmode="country names",
    color="win_prob",
    hover_name="team",
    color_continuous_scale="OrRd",
    title="<b>Géographie des Favoris - WC 2026</b>",
    labels={'win_prob': 'Chances de titre'}
)
fig_map.show()

/var/folders/_j/t0knsx_d5_z7c38r3jzmx7rm0000gn/T/ipykernel_94557/3617330382.py:1: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig_map = px.choropleth(


In [54]:
import joblib
import pandas as pd
import plotly.express as px
import numpy as np

# 1. Chargement du modèle
model = joblib.load("../data/processed/models/champion_xgboost.pkl")

# 2. Récupérer le premier modèle et son pipeline de transformation
first_fold = model.calibrated_classifiers_[0].estimator
preprocessor = first_fold.steps[0][1]  # Le transformer (ColumnTransformer / Encoder)
xgboost_model = first_fold.steps[-1][1] # Le modèle XGBoost

# 3. Tenter de récupérer les noms de colonnes générés
try:
    # Si ton pipeline supporte get_feature_names_out (Scikit-learn moderne)
    feature_names_transformed = preprocessor.get_feature_names_out()
    print(f"✅ {len(feature_names_transformed)} noms de colonnes récupérés du preprocessor.")
except:
    print("⚠️ Impossible de récupérer les noms via get_feature_names_out.")
    # Fallback : on crée des noms génériques pour ne pas bloquer
    feature_names_transformed = [f"feature_{i}" for i in range(len(xgboost_model.feature_importances_))]

# 4. Extraction de l'importance
importances = xgboost_model.feature_importances_

# 5. Création du DataFrame (en s'assurant que les longueurs matchent)
min_len = min(len(feature_names_transformed), len(importances))
df_imp = pd.DataFrame({
    'Feature': feature_names_transformed[:min_len],
    'Importance': importances[:min_len]
}).sort_values('Importance', ascending=False)

print(df_imp.head(20))
# 6. Plot
fig = px.bar(
    df_imp.head(20), 
    x='Importance', 
    y='Feature', 
    orientation='h',
    title="<b>Top 20 : Ce que ton modèle regarde vraiment</b>",
    color='Importance',
    color_continuous_scale='Magma'
)
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

✅ 79 noms de colonnes récupérés du preprocessor.
                                  Feature  Importance
21                  num__fifa_points_diff    0.027651
30       num__home_squad_playing_time_min    0.025474
76       cat__away_confederation_CONMEBOL    0.023857
20                    num__fifa_rank_diff    0.019178
55        num__away_squad_performance_ast    0.017567
44  num__home_squad_per_90_minutes_g_a_pk    0.017318
66  num__away_squad_per_90_minutes_g_a_pk    0.017257
40     num__home_squad_per_90_minutes_gls    0.017179
38       num__home_squad_performance_crdy    0.016918
57       num__away_squad_performance_g_pk    0.016829
62     num__away_squad_per_90_minutes_gls    0.016808
36         num__home_squad_performance_pk    0.016552
63     num__away_squad_per_90_minutes_ast    0.015819
37      num__home_squad_performance_pkatt    0.015635
5                           num__elo_diff    0.015278
35       num__home_squad_performance_g_pk    0.015140
24                   num__home_sq